# Predictive maintenance — de la eșantioane la setul de antrenare

**Scop.** Construiește tabelul de caracteristici pentru detectorul de anomalii (RCF) din
sesiunile geamănului MuJoCo. Notebook-ul este driver-ul: ferestrele și caracteristicile stau în
`cncdt_ml`, ca să ruleze identic în CI, în Studio și într-un job.

**Intrări.** `simulation/mujoco/runs/` local, ori prefixul `training-data/mujoco/` din bucket.
**Ieșiri.** `ml/artifacts/pm/windows.parquet`, `features.parquet` și `pm-dataset-summary.json`
(ignorate de git).

**Regulă.** Logica nouă care merită repetată se mută în `cncdt_ml` și primește test; celula
notebook-ului rămâne subțire.


## 1. Mediu și importuri

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "ml" / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "ml" / "src").is_dir():
    raise RuntimeError("rulează notebook-ul din repository")
sys.path.insert(0, str(ROOT / "ml" / "src"))
print("repository:", ROOT)

from cncdt_ml.mujoco_corpus import (
    build_windows,
    load_events,
    session_directories,
    session_info,
)
from cncdt_ml.offline import build_offline_features


## 2. Configurare

Schimbă `RUN` când vrei altă colectare. Dacă directorul nu există, notebook-ul ia toate
sesiunile de sub `runs/`.


In [ ]:
RUNS = ROOT / "simulation" / "mujoco" / "runs"
RUN = RUNS / "run-20261015T172741"
if not RUN.is_dir():
    RUN = RUNS
WINDOW_S = 60
ARTIFACTS = ROOT / "ml" / "artifacts" / "pm"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("rulare:", RUN)
print("fereastră:", WINDOW_S, "s")


## 3. Auditul sesiunilor

Ce declară fiecare `session.json`: familia, sămânța, numărul de cicluri și totalurile.


In [ ]:
sessions = session_directories(RUN)
summary = [session_info(directory).record() for directory in sessions]
print(f"{len(sessions)} sesiuni")
print(json.dumps(summary, indent=2, ensure_ascii=False))


## 4. Ferestre de 60 s

`build_windows` grupează eșantioanele pe minut UTC, calculează agregările din
`cncdt_ml.features` și păstrează doar coloanele din contract.


In [ ]:
windows = []
for directory in sessions:
    windows.extend(build_windows(directory, window_s=WINDOW_S))
print(f"{len(windows)} ferestre de {WINDOW_S} s")
if windows:
    print(json.dumps(windows[0], indent=2, ensure_ascii=False))


## 5. Caracteristicile offline

Rapoarte derivate, ferestre glisante, decalaje și etichete, în aceeași ordine ca rețeta
Data Wrangler și ca `build_features.py`.


In [ ]:
features = build_offline_features(windows)
print(f"{len(features)} rânduri, {len(features[0]) if features else 0} coloane")


## 6. Etichete: ce se poate folosi acum

`health_label` din `cncdt_ml.labels` este calibrat pe scara celulei Isaac Sim. În MuJoCo,
debitul de coolant mediu pe o fereastră este ~1,8 LPM fiindcă mașina stă mult în `IDLE`, iar
pragul generic de 3 LPM ar eticheta aproape orice fereastră ca `coolant`.

Pentru baseline folosim `twin_label`, calculat în `build_windows` din defectul majoritar al
pieselor, uzura maximă a sculei, frecarea maximă și temperatura maximă. Coolantul rămâne în
afara etichetei până la o calibrare pe ferestre `CUTTING`. Diferența dintre cele două seturi
de etichete se vede mai jos și este un rezultat, nu o eroare de notebook.


In [ ]:
from collections import Counter

twin_labels = Counter(row["twin_label"] for row in features)
contract_labels = Counter(row.get("health_label", "missing") for row in features)
healthy_count = sum(1 for row in features if row["healthy"])
print("twin_label:", dict(sorted(twin_labels.items())))
print("health_label (generic):", dict(sorted(contract_labels.items())))
print("ferestre sănătoase:", healthy_count, "din", len(features))
if healthy_count == 0:
    print("ATENȚIE: rularea nu are ferestre sănătoase; un detector RCF are nevoie de o bază normală.")


## 7. Verificări înainte de antrenare

In [ ]:
required = ("ts_ms", "window_start_ms", "spindle_load_mean", "vibration_rms", "temp_c_mean")
missing = [name for name in required if features and name not in features[0]]
print("coloane lipsă:", missing)

keys = [(row.get("session"), row.get("window_key")) for row in features]
print("ferestre duplicate:", len(keys) - len(set(keys)))

events = [(directory.name, row) for directory in sessions for row in load_events(directory)]
print("evenimente:", len(events), dict(Counter(row.get("event") for _, row in events)))


## 8. Scrie artefactele

Parquet cere `pyarrow`; sumarul JSON se scrie mereu. Artefactele rămân în `ml/artifacts/`,
ignorat de git.


In [ ]:
summary_path = ARTIFACTS / "pm-dataset-summary.json"
summary_path.write_text(
    json.dumps(
        {
            "run": str(RUN),
            "window_s": WINDOW_S,
            "sessions": summary,
            "windows": len(windows),
            "features": len(features),
            "twin_labels": dict(sorted(twin_labels.items())),
            "contract_labels": dict(sorted(contract_labels.items())),
            "healthy_windows": healthy_count,
            "events": len(events),
        },
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)
print("scris", summary_path)

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError:
    print('pyarrow lipsește; rulează `pip install -e "ml[data]"` pentru Parquet')
else:
    pq.write_table(pa.Table.from_pylist(windows), ARTIFACTS / "windows.parquet", compression="snappy")
    pq.write_table(pa.Table.from_pylist(features), ARTIFACTS / "features.parquet", compression="snappy")
    print("scris", ARTIFACTS / "windows.parquet")
    print("scris", ARTIFACTS / "features.parquet")


## 9. Pasul următor

Deschide `pm-02-rcf.ipynb` pentru baseline-ul de detecție. Când baseline-ul se stabilizează,
mută antrenarea în `ml/training/predictive_maintenance/train.py` și păstrează notebook-ul pentru
rapoarte și comparații.
